In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/student_grade_ses_dataset.csv")
df.head()

,student_id,academic_year,survey_date,age,gender,grade_level,school_type,residence_type,family_size,num_siblings,...,distance_to_school_km,scholarship_status,extra_tutoring,parental_involvement_score,study_hours_per_week,attendance_rate_pct,previous_year_grade_pct,final_grade_pct,grade_class,at_risk_flag
0,STU101529,2024-2025,2025-03-06,16,Male,12,Government,Rural,3,0,...,4.0,No Scholarship,0,4,15.0,76.8,61.7,59.4,D,0
1,STU107546,2022-2023,2025-05-02,15,Male,10,Government,Urban,3,1,...,0.7,No Scholarship,0,3,20.9,74.5,49.2,60.2,C,0
2,STU103737,2024-2025,2025-03-21,14,Male,9,Government,Urban,2,0,...,2.9,No Scholarship,0,4,20.9,94.9,56.9,58.4,D,0
3,STU108172,2023-2024,2025-03-30,13,Male,8,Government,Rural,6,4,...,7.0,No Scholarship,0,1,6.5,63.6,49.3,57.0,D,0
4,STU110633,2023-2024,2025-05-06,11,Female,6,Private-Unaided,Rural,5,3,...,9.1,No Scholarship,0,5,13.9,86.0,79.3,88.7,B,0


In [3]:
X = df.drop(columns=["student_id", "academic_year", "survey_date", "grade_level"])
y = df["grade_level"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((9620, 25), (2405, 25), (9620,), (2405,))

In [4]:
preprocessor = ColumnTransformer([
    ('numeric', num_pipe(), ["age", "family_size", "family_size", "num_siblings",  "distance_to_school_km", "extra_tutoring", "parental_involvement_score", "study_hours_per_week", "attendance_rate_pct", "previous_year_grade_pct", "final_grade_pct"]),
    ('categoric', cat_pipe(encoder='onehot'), ["gender", "school_type", "residence_type", "scholarship_status", "grade_class", "at_risk_flag"]),
])

In [5]:
from xgboost import XGBRegressor
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', XGBRegressor(n_jobs=-1, random_state=42))
])

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.xgb_params, cv=3, scoring='r2', n_iter=50, n_jobs=-1, verbose=1, random_state=42)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
{'algo__colsample_bytree': np.float64(0.8458637582367365), 'algo__gamma': 10, 'algo__learning_rate': np.float64(0.852139338138361), 'algo__max_depth': 9, 'algo__n_estimators': 128, 'algo__reg_alpha': np.float64(1.6172900811143165), 'algo__reg_lambda': np.float64(0.001987021538542862), 'algo__subsample': np.float64(0.7934434683002587)}
0.8675908446311951 0.8667240341504415 0.863903820514679
